In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

def rmse(y_train, y):
    diff = y_train-y
    diff2 = diff*diff
    res = np.sqrt(diff2.sum() / len(diff2))
    return res

def find_coefs_lin_regretion(X, y, r = 0.001):
    onec = np.ones(X.shape[0])
    X = np.column_stack([onec, X])
    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])
    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv.dot(X.T).dot(y)
    #w = (XTX)-1 XTy
    return w[0], w[1:]

def prepare_X(df, cats):
    df_copy = df.copy()
    features = base.copy()
        
    for k, vals in cats.items():
        for v in vals:
            df_copy['%s_%s' % (k, v)] = (df_copy[k] == v).astype('int')
            features.append('%s_%s' % (k, v))
    # 
    
    df_copy['age'] = 2017 - df_copy['year']
    
    features.append('age')
    
    for v in [2, 3, 4]:
        df_copy['num_dors_%s' %v] = (df_copy['number_of_doors'] == v).astype('int')
        features.append('num_dors_%s' %v)
        
    df_num = df_copy[features]
    df_num = df_num.fillna(0)
    X = df_num.values
    return X 

url = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-02-car-price/data.csv'

df = pd.read_csv(url)
df.head()

df.columns = df.columns.str.lower().str.replace(" ", "_")
strings = list(df.select_dtypes(include=['str', 'object']))
for col in strings:
    df[col] = df[col].str.lower().str.replace(" ", "_")
log_price = np.log1p(df.msrp)
#sns.histplot(log_price, bins = 50) 
df.isnull().sum()

# validation
n = len(df)
test_data = int(n*0.2)
val_data = int(n*0.2)
train_data = n - (test_data+val_data)

np.random.seed(2)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:train_data]]
df_val = df.iloc[idx[train_data:train_data+val_data]]
df_test = df.iloc[idx[train_data+val_data:]]

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = np.log1p(df_train.msrp.values)
y_val = np.log1p(df_val.msrp.values)
y_test = np.log1p(df_test.msrp.values)

del df_train['msrp']
del df_val['msrp']
del df_test['msrp']

base = ['engine_hp', 'engine_cylinders', 'highway_mpg',
        'city_mpg', 'popularity']
categories_vars = ['make', 'model', 'engine_fuel_type', 'transmission_type', 
                   'driven_wheels', 'market_category', 'vehicle_size', 'vehicle_style']
cats={}
for c in categories_vars:
    cats[c] = list(df_train[c].value_counts().head().index)

# sns.histplot(data=y_pred, bins = 50, alpha = 0.5, color="red")
# sns.histplot(data=y_train, bins = 50, alpha = 0.5, color="blue")



df_full_train = pd.concat([df_train, df_val])
df_full_train = df_full_train.reset_index(drop=True)
y_full = np.concatenate([y_train, y_val])
X_full = prepare_X(df_full_train, cats)
w0, w = find_coefs_lin_regretion(X_full, y_full, 0.001)
X_test = prepare_X(df_test, cats)
y_pred = w0 + X_test.dot(w)

y_val_dol = np.expm1(y_test)
y_pred_dol = np.expm1(y_pred)
print(f"Помилка оцінки вартості автомобіля даної моделі +-{np.abs(y_val_dol - y_pred_dol).mean():.0f}$")
print(f"RMSE = {rmse(y_test, y_pred)}")

car = df_test.iloc[20].to_dict()
price = y_test[20]
df_small = pd.DataFrame([car])
X_small = prepare_X(df_small, cats)
y_pred_price = w0 + X_small.dot(w)
print(np.expm1(y_pred_price))
print(np.expm1(price))
print(car)




Помилка оцінки вартості автомобіля даної моделі +-13135$
RMSE = 0.44763897313756634
[43176.61606111]
{'make': 'toyota', 'model': 'sienna', 'year': 2015, 'engine_fuel_type': 'regular_unleaded', 'engine_hp': 266.0, 'engine_cylinders': 6.0, 'transmission_type': 'automatic', 'driven_wheels': 'front_wheel_drive', 'number_of_doors': 4.0, 'market_category': nan, 'vehicle_size': 'large', 'vehicle_style': 'passenger_minivan', 'highway_mpg': 25, 'city_mpg': 18, 'popularity': 2031}
35000.00000000001
